# שיעור 08 - תבנית עיצוב רב-סוכנית


## הגדרה


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## מדוע מערכות מרובות סוכנים?

משימות מהחיים האמיתיים כמו תכנון טיול כוללות סוגים רבים ושונים של מומחיות — לוגיסטיקה, ידע מקומי, תקצוב, ועוד. סוכן בודד שמנסה לטפל בכל quickly הופך במהירות לבלתי נוח.

מערכות מרובות סוכנים פותרות זאת באמצעות **התמחות**: כל סוכן מתמקד בתחום מומחיות אחד, ומפיק תוצאות באיכות גבוהה יותר מאשר כללי. הן גם משפרות את ה-**יכולת להתרחב** — אפשר להוסיף סוכנים חדשים (למשל, מומחה טיסות, מבקר מסעדות) בלי לכתוב מחדש את תהליך העבודה הקיים. הסוכנים מורכבים יחד דרך צינור מובנה, שמעביר הקשר מאחד לשני.


## יצירת סוכנים מותאמים


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## בניית זרימת עבודה עוקבת

`WorkflowBuilder` מאפשרת לך לחבר סוכנים לגרף מכוון. כאן אנו יוצרים צינור דו-שלבי פשוט: ה-**TravelPlanner** מתווה את מסלול הטיול, ולאחר מכן ה-**TravelConcierge** בודק ומשפר אותו.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## הוספת סוכנים נוספים לזרימת העבודה

אחת מהיתרונות הגדולים בדפוס רב-הסוכנים היא כמה קל להרחיב אותו. למטה נוסיף סוכן **BudgetReviewer** שבודק את התוכנית מול התקציב של המטייל, מסמן פריטים שעשויים לדחוף עלויות מעבר למגבלה, ומציע חלופות לחיסכון בכסף. זרימת העבודה מפעילה כעת שלושה סוכנים ברצף:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## סיכום

בשיעור זה למדת כיצד:

1. **ליצור סוכנים מומחים** — כל אחד עם תפקיד ממוקד (תכנון, מנהל קונסיירז', סקירת תקציב).
2. **לחבר סוכנים לזרימת עבודה סדרתית** באמצעות `WorkflowBuilder` ו-`add_edge`.
3. **לשדר פלט** מצינור רב-סוכנים, ולעקוב אחר הסוכן המדבר.
4. **להרחיב זרימת עבודה** על ידי הוספת סוכנים חדשים לשרשרת מבלי לשנות את הקיימים.

תבנית העיצוב רב-סוכנית שומרת על פשטות כל סוכן, תוך הפקת תוצאות עשירות ומבוקרות יותר מאשר כל סוכן בודד יכול להשיג לבד.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
